In [ ]:
# ============================================================
# PARAMETERS (papermill injects these)
# ============================================================

config_path = None
run_dir = None

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import json
import yaml
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from pathlib import Path

from collections import Counter

from scripts.set_seed import set_seed

from src.model_factory import build_model

from src.dataset import CLASS_MAP

from src.embedding_registry import get_embedding_dir

from src.embeddings import load_single_embeddings_from_manifest

from src.training import (
    train_classifier,
    make_weighted_ce,
    print_final_training_summary,
    summarize_final_in_sample_metrics,
    final_in_sample_classification_table,
    plot_final_training_history,
    plot_final_macro_metrics,
    plot_final_confusion_matrix,
    plot_final_roc_curves,
    evaluate_split,
)

In [ ]:
# ============================================================
# LOAD CONFIG, SEED, AND MODEL
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

RUN_DIR = Path(run_dir)
RUN_DIR.mkdir(parents=True, exist_ok=True)

set_seed(cfg["experiment"]["seed"])

model = build_model(cfg)

print(json.dumps(cfg, indent=2))

In [ ]:
# ============================================================
# LOAD EMBEDDINGS
# ============================================================

evaluation_mode = cfg["evaluation"]["mode"]

evaluation_strat = cfg["data"]["split_strategy"] if evaluation_mode == "single" and cfg["embedder"]["use_msa_mode"] else "N.A."

print(f"Evaluation mode: {evaluation_mode}")
print(f"Evaluation strategy: {evaluation_strat}")

embedding_manifest = get_embedding_dir(cfg)

print(f"Using embedding manifest: {embedding_manifest}")

embeddings_dict = load_single_embeddings_from_manifest(
    manifest_candidates=[str(embedding_manifest)]
)

In [ ]:
# ============================================================
# FINAL MODEL TRAINING
# ============================================================

# Prepare all-data loaders
train_ds = TensorDataset(
    embeddings_dict["train_embeddings"], embeddings_dict["train_labels"]
)
val_ds = TensorDataset(embeddings_dict["val_embeddings"], embeddings_dict["val_labels"])

if evaluation_strat in ["diverse", "family", "balanced"]:
    is_chunked = True
    batch_size = embeddings_dict.get("chunk_size")
else:
    is_chunked = False
    batch_size = 128

if cfg["hyperparameters"]["use_custom_hyperparameters"]:
    best_hp = cfg["hyperparameters"]
    class_counts_all = Counter(embeddings_dict["train_labels"].numpy())
    counts_list_all = [class_counts_all.get(i, 1) for i in range(3)]

    if best_hp["useWeightedSampler"]:
        sample_weights_all = torch.tensor(
            [
                1.0 / class_counts_all.get(int(l), 1)
                for l in embeddings_dict["train_labels"].numpy()
            ],
            dtype=torch.float,
        )
        sampler_all = WeightedRandomSampler(
            sample_weights_all,
            len(sample_weights_all),
            replacement=True,
        )
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size if is_chunked else int(best_hp["batch_size"]),
            sampler=sampler_all,
            drop_last=True,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size if is_chunked else int(best_hp["batch_size"]),
            shuffle=True,
            drop_last=True,
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size if is_chunked else int(best_hp["batch_size"]),
        shuffle=False,
        drop_last=False,
    )

    criterion = (
        make_weighted_ce(
            counts_list_all, device, power=float(best_hp.get("weighted_ce_power", 1.0))
        )
        if float(best_hp.get("weighted_ce_power", -1.0)) >= 0.0
        else None
    )

else:
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False) # shuffle False because already shuffled once while generating embeddings
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    criterion = None
    best_hp = {}

# Train model
embedding_dim = (
    int(embeddings_dict["train_embeddings"].shape[-1])
    if embeddings_dict["train_embeddings"].numel()
    else 768
)
model_init_args = dict(cfg.get("model", {}).get("init_args", {}))
model_init_args.setdefault("embedding_dim", embedding_dim)

final_model = model(**model_init_args).to(device)

print(f"Final model initialized with args: {model_init_args} and batch size: {batch_size}")
history = train_classifier(
    final_model,
    train_loader,
    val_loader,
    num_epochs=50,
    device=device,
    patience=50,
    criterion=criterion,
    optimizer_lr=float(best_hp.get("optimizer_lr", 1e-3)),
    optimizer_weight_decay=float(best_hp.get("optimizer_weight_decay", 1e-2)),
    schedular_patience=int(best_hp.get("schedular_patience", 10)),
    warmup_epochs=int(best_hp.get("warmup_epochs", 5)),
    checkpoint_path=str(RUN_DIR / "model_best_epoch.pt"),
)

In [ ]:
# Display tables
print_final_training_summary(
    history, save_path=RUN_DIR / "final_training", insample=False
)
summarize_final_in_sample_metrics(
    history,
    class_names=["barrier", "cation", "anion"],
    save_path=RUN_DIR / "final_training",
    insample=False,
)
final_in_sample_classification_table(
    history,
    class_names=["barrier", "cation", "anion"],
    save_path=RUN_DIR / "final_training",
    insample=False,
)

# Display plots
plot_final_training_history(
    history, save_path=RUN_DIR / "final_training", insample=False
)
plot_final_macro_metrics(history, save_path=RUN_DIR / "final_training", insample=False)
plot_final_confusion_matrix(
    history,
    class_names=["barrier", "cation", "anion"],
    save_path=RUN_DIR / "final_training",
    insample=False,
)
plot_final_roc_curves(
    history,
    class_names=["barrier", "cation", "anion"],
    save_path=RUN_DIR / "final_training",
    insample=False,
)

In [ ]:
# ============================================================
# SAVE FINAL MODEL
# ============================================================

torch.save(
    final_model.state_dict(),
    RUN_DIR / "final_model.pt",
)

print("Final model saved.")

In [ ]:
# ============================================================
# UNSEEN DATA EVALUATION
# ============================================================

if evaluation_strat in ["diverse", "family", "balanced"]:
    unseen_manifest = get_embedding_dir(cfg, unseen=True)
    unseen_dict = load_single_embeddings_from_manifest(manifest_candidates=[str(unseen_manifest)])
    n_unseen = unseen_dict.get("chunk_size")
    
    print(f"Evaluating on unseen split ({n_unseen} samples)")

    unseen_embeddings = unseen_dict["val_embeddings"].to(device)
    unseen_labels = unseen_dict["val_labels"].cpu().numpy()
    unseen_sequence_ids = unseen_dict["val_sequence_ids"]
    unseen_seqs = unseen_dict["val_seqs"]

    final_model.eval()
    with torch.no_grad():
        logits = final_model(unseen_embeddings)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1).cpu().numpy()

    for i, cls in enumerate(preds):
        print(f"\n({i}) {unseen_sequence_ids[i]}:")
        print(f"    Sequence: {unseen_seqs[i]}")
        print(f"    Predicted class: {CLASS_MAP[cls]}, confidence={probs[i, cls]:.3f}")
        print(f"    True class: {CLASS_MAP[unseen_labels[i]]}")

    # Some numeric metrics
    print("\nFinal evaluation on unseen split:")
    evaluate_split(
        "Unseen Test",
        final_model,
        unseen_embeddings,
        unseen_labels,
        class_names=["barrier", "cation", "anion"],
    )

    # Save results along with final metrics
    results_df = pd.DataFrame({
        "sequence_id": unseen_sequence_ids,
        "sequence": unseen_seqs,
        "true_label": [CLASS_MAP[l] for l in unseen_labels],
        "predicted_label": [CLASS_MAP[c] for c in preds],
        "confidence": [probs[i, c].item() for i, c in enumerate(preds)],
    })
    results_df.to_csv(RUN_DIR / "final_training" / "unseen_evaluation_results.csv", index=False)



In [ ]:
# ============================================================
# SAVE FINAL METADATA
# ============================================================

final_metadata = {
    "experiment_name": cfg["experiment"]["name"],
    "evaluation_mode": evaluation_mode,
}

with open(
    RUN_DIR / "training_metadata.json",
    "w",
) as f:
    json.dump(final_metadata, f, indent=2)

print("Training complete.")